In [6]:
# Core libraries
from transformers import pipeline
import textwrap
import re

print("Libraries imported successfully!")

Libraries imported successfully!


In [7]:
# ── Model Configuration ──────────────────────────────────────
MODEL_NAME     = "gpt2-medium"   # Pre-trained model from Hugging Face Hub
MAX_NEW_TOKENS = 130             # Max tokens to generate per response
WRAP_WIDTH     = 80              # Console output wrap width

print(f"Loading '{MODEL_NAME}' model... (first run may take a moment)\n")

# Load text-generation pipeline
# device=-1 means CPU; set device=0 to use GPU if available
generator = pipeline(
    "text-generation",
    model  = MODEL_NAME,
    device = -1,
)

print(f"Model '{MODEL_NAME}' loaded successfully!")

Loading 'gpt2-medium' model... (first run may take a moment)



Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Model 'gpt2-medium' loaded successfully!


In [8]:
# ── System Context (Prompt Prefix) ───────────────────────────
# This tells GPT-2 to behave like a knowledgeable assistant
SYSTEM_CONTEXT = (
    "You are a knowledgeable and helpful AI assistant. "
    "Answer questions clearly, accurately and concisely.\n\n"
)

def build_prompt(question: str) -> str:
    """
    Constructs the full prompt from user question.
    GPT-2 will complete the 'Answer:' section.
    """
    return f"{SYSTEM_CONTEXT}Question: {question}\nAnswer:"

# ── Test the prompt builder ───────────────────────────────────
sample_prompt = build_prompt("What is Artificial Intelligence?")
print("Sample Prompt:")
print("-" * 50)
print(sample_prompt)
print("-" * 50)

Sample Prompt:
--------------------------------------------------
You are a knowledgeable and helpful AI assistant. Answer questions clearly, accurately and concisely.

Question: What is Artificial Intelligence?
Answer:
--------------------------------------------------


In [9]:
def extract_answer(generated: str, prompt: str) -> str:
    """
    Strips the prompt prefix from generated text
    and returns only the clean answer portion.
    """
    # Remove the prompt we passed in
    answer = generated[len(prompt):].strip()

    # Keep only the first 3 complete sentences
    sentences = re.split(r'(?<=[.!?])\s+', answer)
    clean = " ".join(sentences[:3]).strip()

    # Remove any incomplete trailing sentence
    if clean and clean[-1] not in ".!?":
        clean = clean.rsplit(".", 1)[0] + "." if "." in clean else clean

    return clean if clean else "I don't have enough information to answer that."


def get_response(user_question: str) -> str:
    """
    Full pipeline:
        1. Build prompt from user question
        2. Run GPT-2 text generation
        3. Extract and clean the answer
    """
    # Step 1 — Build prompt
    prompt = build_prompt(user_question)

    # Step 2 — Generate response
    result = generator(
        prompt,
        max_new_tokens       = MAX_NEW_TOKENS,
        num_return_sequences = 1,
        do_sample            = True,     # Enables sampling for diverse output
        top_k                = 40,       # Limits vocabulary to top 40 tokens
        top_p                = 0.90,     # Nucleus sampling threshold
        temperature          = 0.70,     # Controls randomness (lower = focused)
        repetition_penalty   = 1.4,      # Penalises repeated phrases
        pad_token_id         = 50256,    # GPT-2 EOS token ID
    )

    # Step 3 — Extract clean answer
    raw = result[0]["generated_text"]
    return extract_answer(raw, prompt)


print("Response generation functions defined!")

Response generation functions defined!


In [10]:
# ── Quick Test ───────────────────────────────────────────────
test_questions = [
    "What is Artificial Intelligence?",
    "Who created Python?",
    "What is machine learning?",
]

print("Running test queries...\n")
print("=" * 55)

for q in test_questions:
    print(f"You    : {q}")
    answer = get_response(q)
    wrapped = textwrap.fill(
        answer,
        width              = WRAP_WIDTH,
        initial_indent     = "Chatbot: ",
        subsequent_indent  = "         "
    )
    print(wrapped)
    print("-" * 55)

Both `max_new_tokens` (=130) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running test queries...

You    : What is Artificial Intelligence?


Both `max_new_tokens` (=130) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Chatbot: This website provides information on artificial intelligence (AI). It
         has been created to help you understand the concepts of advanced
         algorithms such as Deep Learning. The purpose behind this site was not
         only for educational purposes but also because it can provide many
         benefits in your life if used properly.
-------------------------------------------------------
You    : Who created Python?


Both `max_new_tokens` (=130) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Chatbot: It was inspired by the programming language C++. The name of this tool
         is derived from "Python", which stands for Computer Science in English.
         In essence it helps you to write code faster with fewer bugs while
         maintaining better quality results than other programs available on the
         market today (for example GCC).
-------------------------------------------------------
You    : What is machine learning?
Chatbot: Machine Learning refers to the process of building complex algorithms
         that can be applied in many different fields like medicine or robotics.
         It has been used for some time by companies such as Google (Google
         Brain), Amazon Alexa with its voice command technology, Microsoft
         Cortana using artificial intelligence capabilities on Windows 10 PCs
         etc..
-------------------------------------------------------


In [11]:
# ── Main Chatbot Loop ────────────────────────────────────────
def run_chatbot():
    print("=" * 55)
    print("Knowledge Q&A Bot  (GPT-2)")
    print("=" * 55)
    print("Chatbot: Hello! I am your AI Knowledge Assistant.")
    print("         Ask me anything and I'll do my best to help!")
    print("         (type 'exit' or 'quit' to end)\n")

    while True:
        # ── Accept User Input ────────────────────────────────
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nChatbot: Goodbye! Keep exploring knowledge! ")
            break

        # ── Skip empty input ─────────────────────────────────
        if not user_input:
            continue

        # ── Exit Condition ───────────────────────────────────
        if user_input.lower() in {"exit", "quit"}:
            print("Chatbot: It was great helping you! Goodbye! ")
            break

        # ── Generate and Display Response ────────────────────
        response = get_response(user_input)

        wrapped = textwrap.fill(
            response,
            width             = WRAP_WIDTH,
            initial_indent    = "Chatbot: ",
            subsequent_indent = "         "
        )
        print(f"{wrapped}\n")


# ── Start the Chatbot ────────────────────────────────────────
run_chatbot()

   📚  Knowledge Q&A Bot  (GPT-2)
Chatbot: Hello! I am your AI Knowledge Assistant.
         Ask me anything and I'll do my best to help!
         (type 'exit' or 'quit' to end)



Both `max_new_tokens` (=130) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Chatbot: Hello! I am very sorry to hear about your recent troubles with the
         botnet you have created for me at work. However we will try our best
         not too much so that it is able be activated on time as well - please
         don't worry if anything goes wrong during activation of this service or
         even after deployment/activation :)   (Note from my team : We're
         working hard behind-the scenes in order create an extremely efficient
         network infrastructure which supports all aspects including bandwidth
         usage etc.



Both `max_new_tokens` (=130) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Chatbot: CPPMP – Computer Programmer Professional Package. It allows you to
         write programs in Python with the help of an expert programmer from our
         team at LISNERIA Software Solutions Inc., based out Ofenbosch South
         Africa (USA). We will explain how it works, what we do for free after
         your exam which may be very useful!



Both `max_new_tokens` (=130) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Chatbot: panda as in the animal that can read people's minds? This machine has
         been developed by IBM to help you understand human language better! It
         will also give you feedback on your work if it fails or doesn't solve
         an issue correctly (for example when using complex sentence structure).

Chatbot: It was great helping you! Goodbye! 👋
